# Session 4: Scraping with Beautiful Soup

We will scrape **https://books.toscrape.com/** and turn one webpage into a Pandas dataframe.

By the end, you should be able to:

- request a webpage
- check whether the request worked
- parse HTML
- identify a repeated container
- extract titles, prices, ratings, availability and links
- create and inspect a dataframe
- clean and save the scraped data

> Workflow: **Request → Parse → Select → Extract → Store → Validate**

## Before scraping

Understand the site's html structure.

## 1. Install packages if needed

### This cell installs the three external libraries used in the notebook:

- `requests` downloads the webpage.
- `beautifulsoup4` reads and searches the HTML.
- `pandas` turns the extracted information into a table.

In [1]:
# !python3 -m pip install requests beautifulsoup4 pandas

## 2. Import libraries

### We are loading four tools:

- `requests` will fetch the webpage.
- `BeautifulSoup` will turn raw HTML into a searchable structure.
- `pandas`, shortened to `pd`, will create and analyse our dataframe.
- `urljoin` will turn incomplete links from the webpage into full URLs.

Running an import makes these tools available to the notebook.

In [2]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
from urllib.parse import urljoin

#### import requests
A Python library that sends requests to websites and downloads their content. Without it, Python has no way to ask a website for its contents.

#### from urllib.parse import urljoin
A function that combines a website's base URL with an incomplete (relative) link to create a full webpage address. We use it because many websites store links as relative paths instead of complete URLs.


Website
   │
   ▼
requests
(download page)
   │
   ▼
Beautiful Soup
(read HTML)
   │
   ▼
urljoin
(fix incomplete links)
   │
   ▼
Pandas
(store everything as a dataframe)

## 3. Store the URL

### We are saving the webpage address inside a variable called `url`.

The URL is text, so it must be placed inside quotation marks. Storing it in a variable means we can reuse the address later without typing it repeatedly.

The second line displays the value so we can confirm that the variable contains the expected address.

In [7]:
url="https://books.toscrape.com/"

## 4. Request the webpage

This is the first time Python communicates with a website.

When we write:

```python
response = requests.get(url)
```

the following happens:

1. Python sends an HTTP **GET request** to the website.
2. The website's server receives that request.
3. The server sends back a **response**.
4. We store that response in a variable called `response`.

Think of it like ordering food:

- **You** → make the request.
- **Restaurant** → prepares the order.
- **Delivery bag** → the response.

The response contains much more than the webpage itself. It also contains:
- the status code
- headers
- cookies
- and the HTML that generated the page.

We haven't downloaded a spreadsheet—we've downloaded the webpage itself.


### What we are about to do

This cell sends an HTTP request to the website.

- `requests.get(url, timeout=30)` asks the server for the page stored in `url`.
- `timeout=30` prevents Python from waiting forever if the website does not respond.
- The server's reply is saved in a variable called `response`.
- Writing `response` on the final line displays a short summary, usually something such as `<Response [200]>`.

At this stage, `response` contains the status code, headers and webpage HTML.

In [8]:
response = requests.get(url,timeout=30)
response

<Response [200]>

### Optional: Check the response's status code.

A status code is the server's short message about what happened:

- `200` means the request succeeded.
- `403` means access was refused.
- `404` means the page was not found.
- `500` means the server encountered an error.

We should check this before trying to parse the page.

## 5. Look at the raw HTML

The browser turns HTML into something beautiful using CSS. But, Python doesn't see colours, buttons or layouts.
It only receives the underlying HTML.

The next two lines do two different jobs:

```python
html = response.text
```
stores the webpage's HTML inside a variable called `html`.

```python
print(html[:1000])
```

prints only the **first 1,000 characters**.

Why only the first thousand?

A webpage can contain tens of thousands of characters. Printing everything is a lot.

`[:1000]` is called **slicing** and simply means:

> "Show me the first thousand characters."

### Separate the webpage's HTML from the rest of the response.

- `response.text` contains the HTML as one long string of text.
- We store that string in a variable called `html`.
- `html[:1000]` uses slicing to keep only the first 1,000 characters.
- `print()` displays that shortened sample.

We inspect only the beginning because printing the full page would overwhelm the notebook.

In [9]:
html = response.text 
print(html[:1000])

<!DOCTYPE html>
<!--[if lt IE 7]>      <html lang="en-us" class="no-js lt-ie9 lt-ie8 lt-ie7"> <![endif]-->
<!--[if IE 7]>         <html lang="en-us" class="no-js lt-ie9 lt-ie8"> <![endif]-->
<!--[if IE 8]>         <html lang="en-us" class="no-js lt-ie9"> <![endif]-->
<!--[if gt IE 8]><!--> <html lang="en-us" class="no-js"> <!--<![endif]-->
    <head>
        <title>
    All products | Books to Scrape - Sandbox
</title>

        <meta http-equiv="content-type" content="text/html; charset=UTF-8" />
        <meta name="created" content="24th Jun 2016 09:29" />
        <meta name="description" content="" />
        <meta name="viewport" content="width=device-width" />
        <meta name="robots" content="NOARCHIVE,NOCACHE" />

        <!-- Le HTML5 shim, for IE6-8 support of HTML elements -->
        <!--[if lt IE 9]>
        <script src="//html5shim.googlecode.com/svn/trunk/html5.js"></script>
        <![endif]-->

        
            <link rel="shortcut icon" href="static/oscar/favicon.

Look for repeated structures such as:

```html
<article class="product_pod">
<p class="price_color">£51.77</p>
```

## 6. Parse the HTML

Right now `html` is just one very long string of text. Beautiful Soup converts that text into something we can search.

Think of it like this:

Before Beautiful Soup:
```
One giant wall of text
```

After Beautiful Soup:
```
A searchable tree of elements
```

Instead of searching through thousands of characters ourselves, we can now ask questions like:
- Find the first heading.
- Find every book.
- Find every price.
- Find every link.

### Turning the raw HTML string into a Beautiful Soup object.

- `html` is currently plain text.
- `"html.parser"` tells Beautiful Soup which parser to use.
- The parsed page is stored in `soup`.
- `type(soup)` checks what kind of Python object was created.

After this step, we can search the webpage by tags, classes and other HTML features.

In [10]:
soup = BeautifulSoup(html, "html.parser")
type(soup)

bs4.BeautifulSoup

## 7. Find the page heading

### What we are about to do

We are asking Beautiful Soup to find the first `<h1>` heading on the page.

- `soup.find("h1")` searches the parsed HTML.
- The matching HTML element is stored in `heading`.
- Displaying `heading` shows both the tag and its contents.

This is our first simple test that Beautiful Soup can locate an element successfully.

In [11]:
heading = soup.find_all("h1")
heading

[<h1>All products</h1>]

In [12]:
heading_new = soup.find("h1")
heading_new

<h1>All products</h1>

### The `heading` variable currently contains an HTML element such as `<h1>All products</h1>`.

`get_text(strip=True)` removes the surrounding tags and returns only the visible words. `strip=True` also removes extra spaces and line breaks from the beginning and end.

In [15]:
heading_new.get_text

<bound method PageElement.get_text of <h1>All products</h1>>

In [26]:
heading_new.get_text(strip = True)

'All products'

## 8. Find one book card

Every book lives inside:

```html
<article class="product_pod">
```

Every book has the **same HTML structure**. That repetition is exactly what makes scraping possible.
Rather than writing code twenty times, we teach Python how to recognise one book card. Then Python repeats the same process for every card.

### Locating the first repeated book card.

`article.product_pod` is a CSS selector:

- `article` refers to the HTML tag.
- The period means “class”.
- `product_pod` is the class name shared by every book card.
- `select_one()` returns only the first matching card.

We save that complete card in `first_book` so we can practise extracting one field at a time before looping over all books.

In [19]:
first_book = soup.select_one("article.product_pod")
print(first_book)

<article class="product_pod">
<div class="image_container">
<a href="catalogue/a-light-in-the-attic_1000/index.html"><img alt="A Light in the Attic" class="thumbnail" src="media/cache/2c/da/2cdad67c44b002e7ead0cc35693c0e8b.jpg"/></a>
</div>
<p class="star-rating Three">
<i class="icon-star"></i>
<i class="icon-star"></i>
<i class="icon-star"></i>
<i class="icon-star"></i>
<i class="icon-star"></i>
</p>
<h3><a href="catalogue/a-light-in-the-attic_1000/index.html" title="A Light in the Attic">A Light in the ...</a></h3>
<div class="product_price">
<p class="price_color">Â£51.77</p>
<p class="instock availability">
<i class="icon-ok"></i>
    
        In stock
    
</p>
<form>
<button class="btn btn-primary btn-block" data-loading-text="Adding..." type="submit">Add to basket</button>
</form>
</div>
</article>


`article.product_pod` means:

> Find an `article` element whose class is `product_pod`.


In [20]:
all_books = soup.select("article.product_pod")

In [21]:
all_books

[<article class="product_pod">
 <div class="image_container">
 <a href="catalogue/a-light-in-the-attic_1000/index.html"><img alt="A Light in the Attic" class="thumbnail" src="media/cache/2c/da/2cdad67c44b002e7ead0cc35693c0e8b.jpg"/></a>
 </div>
 <p class="star-rating Three">
 <i class="icon-star"></i>
 <i class="icon-star"></i>
 <i class="icon-star"></i>
 <i class="icon-star"></i>
 <i class="icon-star"></i>
 </p>
 <h3><a href="catalogue/a-light-in-the-attic_1000/index.html" title="A Light in the Attic">A Light in the ...</a></h3>
 <div class="product_price">
 <p class="price_color">Â£51.77</p>
 <p class="instock availability">
 <i class="icon-ok"></i>
     
         In stock
     
 </p>
 <form>
 <button class="btn btn-primary btn-block" data-loading-text="Adding..." type="submit">Add to basket</button>
 </form>
 </div>
 </article>,
 <article class="product_pod">
 <div class="image_container">
 <a href="catalogue/tipping-the-velvet_999/index.html"><img alt="Tipping the Velvet" class="th

## 9. Extract one title

### Inside the first book card, locate the link that contains the title.

The selector `h3 a` means:

> Find an `<a>` link located inside an `<h3>` heading.

We store the matching HTML element in `title_element` and display it so we can inspect its visible text and attributes.

In [22]:
book_name = first_book.select_one("h3")
book_name

<h3><a href="catalogue/a-light-in-the-attic_1000/index.html" title="A Light in the Attic">A Light in the ...</a></h3>

### The full book title is stored in the link's `title` attribute.

- `title_element["title"]` extracts that attribute's value.
- We save it in a variable called `title`.
- The final line displays the extracted title.

This is different from `get_text()`: here we are reading an HTML attribute rather than the visible words between the tags.

- `.get_text()` extracts visible text.
- `["title"]` extracts the value of an HTML attribute.


## 10. Extract the price

### Extracting the price from the first book card.

- `p.price_color` means a `<p>` element whose class is `price_color`.
- `select_one()` finds the first matching price within this book.
- `get_text(strip=True)` removes the HTML tags and unnecessary whitespace.
- The result is stored as `price_text`.

The value still includes the `£` symbol, so it is text rather than a number for now.

In [28]:
price_text = first_book.select_one("p.price_color").get_text(strip=True)
price_text

'Â£51.77'

## 11. Extract availability

### We are extracting the availability message from the first book card.

The selector `p.instock.availability` finds a paragraph that has both classes: `instock` and `availability`.

`get_text(strip=True)` returns only the visible words, such as `In stock`, and stores them in the `availability` variable.

## 12. Extract the rating

### The star rating is not written as visible text. It is encoded in the element's class names.

This cell:

1. Finds the paragraph with class `star-rating`.
2. Stores the element in `rating_element`.
3. Uses `.get("class")` to retrieve its list of classes.

We expect a result such as `["star-rating", "Three"]`.

In [29]:
rating_element = first_book.select_one("p.star-rating")
rating_element.get("class")

['star-rating', 'Three']

### What we are about to do

The class list contains two items:

```python
["star-rating", "Three"]
```

Python counts list positions from zero:

- position `0` is `"star-rating"`
- position `1` is `"Three"`

This cell selects the second item and stores it as the book's rating.

In [30]:
rating = rating_element.get("class")[1]
rating

'Three'

## 13. Extract the link

### The webpage stores a relative link rather than a complete web address. `title_element.get("href")` retrieves that incomplete path.

`urljoin(url, relative_link)` combines the site's base URL with the relative path to create a complete, usable book URL.

## 14. Store one book as a dictionary

### Grouping the first book's extracted fields into a dictionary.

A dictionary stores information as `key: value` pairs:

- the keys will later become dataframe column names
- the values will become the cells in one row

This dictionary represents one complete observation: one book.

In [33]:
first_book_data = {
    "price_text": price_text,
    "rating": rating,
    #"availability": availability,
    #"book_url": book_url
}

first_book_data

{'price_text': 'Â£51.77', 'rating': 'Three'}

Dictionary is very close to one dataframe row:

- dictionary = row
- key = column
- value = cell

## 15. Find all book cards

### Now find **every** book card on the page.

- `select()` returns all matching elements, unlike `select_one()`, which returns only the first.
- The resulting collection is stored in `books`.
- `len(books)` counts how many cards were found.

The webpage shows 20 books, so a result of 20 is an important validation check.

The homepage shows 20 books, so we expect 20 matches.

## 16. Loop through the books

### What we are about to do: understand the loop

`books` contains 20 separate book-card elements. Rather than copying the same title-extraction code 20 times, we use a `for` loop.

Read the first line as:

> For each individual `book` inside the collection called `books`, repeat the indented instructions.

For every card, Python:

1. gives the current card the temporary name `book`
2. searches inside that card for `h3 a`
3. extracts the `title` attribute
4. saves it temporarily as `title`
5. prints the title
6. moves to the next card and repeats

The indentation matters: both indented lines belong to the loop. When the loop finishes, all 20 titles should have been printed.

## 17. Build a list of rows

So far we've extracted information from **one** book.

Now we want **all 20 books**.


1. Create an empty list called `rows`.
2. Visit each book card one at a time.
3. Extract the title, price, rating, availability and link.
4. Store those five pieces of information as a dictionary.
5. Add that dictionary to our growing list.

When the loop finishes, `rows` will contain one dictionary for every book.

Later, Pandas will convert that list directly into a dataframe.

### Build one row per book

This is the main scraping loop.

First, `rows = []` creates an empty list that will collect our results.

Then, for each book card, Python:

1. finds the title link
2. extracts the title
3. extracts the price
4. extracts the rating class
5. extracts availability
6. builds the full book URL
7. creates one dictionary containing those fields
8. appends that dictionary to `rows`

After the loop, `rows` should contain 20 dictionaries—one for every book. `len(rows)` checks that count.

### Inspecting the first dictionary stored in the `rows` list.

Python uses zero-based indexing, so `rows[0]` means “show the first item”. This lets us confirm that the loop stored the expected fields before we create a dataframe.

## 18. Create a dataframe

This is the moment where web scraping meets Pandas.

Currently:

```
rows
```

is a **list of dictionaries**.

Pandas knows how to turn that structure into a table automatically.

Think about the mapping:

- one dictionary → one row
- dictionary keys → column names
- dictionary values → cells

After this line, everything you've already learned in Pandas works exactly the same.


### Convert the list of dictionaries into a Pandas dataframe.

Pandas interprets the structure automatically:

- each dictionary becomes one row
- each dictionary key becomes a column
- each dictionary value becomes a cell

`df.head()` then displays the first five scraped books so we can inspect the result.

### `df.shape` reports the dataframe's dimensions as:

```text
(number of rows, number of columns)
```

We expect 20 rows because the homepage contains 20 book cards. This is another check that our scraper found the expected number of observations.

Ask:

- What does one row represent?
- What does each column represent?
- Did we get the expected 20 rows?
- Are any values missing?


## 19. Clean the price

### The scraped price contains a currency symbol, so Pandas currently treats it as text.

This cell creates a new numeric column:

1. select `price_text`
2. remove the `£` symbol with `.str.replace()`
3. convert the remaining text to decimal numbers using `.astype(float)`
4. save the result as `price_gbp`

We keep the original text column so the transformation remains transparent.

This:

1. selects the price text
2. removes `£`
3. converts the result to a number
4. saves it as a new column


## 20. Convert ratings to numbers

### The ratings are words such as `One`, `Two` and `Five`. We want numeric values that are easier to sort, filter and summarise.

First, we create a dictionary that maps each word to a number. Then `.map(rating_map)` looks up every rating and writes the corresponding number into a new column called `rating_number`.

## 21. Analyse the scraped data

### Count how many books received each rating.

- `value_counts()` counts the frequency of every rating value.
- `sort_index()` arranges the result in rating order from 1 to 5 rather than by frequency.

This shows the distribution of ratings on the first page.

### Sorting books

1. sorts all books by the numeric price column
2. puts the most expensive books first because `ascending=False`
3. selects only the title, price and rating columns
4. displays the first ten rows

This answers a simple reporting-style question: which books on this page are most expensive?

## 22. Validate the scrape

A scraper can run without an error and still collect the wrong data. Compare a few rows manually with the webpage.


## 23. Select final columns

### Create a cleaner final dataframe containing only the columns we want to keep.

`.copy()` makes an independent copy of those selected columns. This is useful because later changes to `books_df` will not accidentally modify the original `df`.

## 24. Save as CSV

## Homework

Answer all three:

1. Which is the cheapest five-star book?
2. How many books cost more than £40?
3. What is the median price?

### Cheapest five-star book

Use the dataframe you created to:

1. filter for books whose `rating_number` is 5
2. sort those books by `price_gbp` from smallest to largest
3. display the first row

Write the code yourself rather than copying the earlier example unchanged.

In [ ]:
# 1. Cheapest five-star book

# Write your code here

### Count books above £40

Create a condition for `price_gbp > 40`, filter the dataframe, and count the matching rows.

You can count the rows using `len()` or by checking the filtered dataframe's shape.

In [ ]:
# 2. Number of books costing more than £40

# Write your code here

### Calculate the median price

Use the numeric `price_gbp` column and the `.median()` method.

The median is the middle price after the values are ordered. It is less affected by unusually expensive or cheap books than the mean.

In [ ]:
# 3. Median price

# Write your code here

### What we are about to do

`response.raise_for_status()` asks Python to stop and produce a clear error if the server returned an unsuccessful status code.

If the request succeeded, Python moves to the next line and prints a confirmation message. This is safer than continuing with an empty or failed response and discovering the problem much later.

In [ ]:
import time

all_rows = []

for page_number in range(1, 4):
    page_url = (
        "https://books.toscrape.com/catalogue/"
        f"page-{page_number}.html"
    )

    page_response = requests.get(page_url, timeout=30)
    page_response.raise_for_status()

    page_soup = BeautifulSoup(
        page_response.text,
        "html.parser"
    )

    page_books = page_soup.select("article.product_pod")

    for book in page_books:
        title_element = book.select_one("h3 a")

        all_rows.append({
            "title": title_element["title"],
            "price_text": book.select_one(
                "p.price_color"
            ).get_text(strip=True),
            "rating": book.select_one(
                "p.star-rating"
            ).get("class")[1],
            "availability": book.select_one(
                "p.instock.availability"
            ).get_text(strip=True),
            "book_url": urljoin(
                page_url,
                title_element.get("href")
            ),
            "source_page": page_number
        })

    time.sleep(1)

multi_page_df = pd.DataFrame(all_rows)
multi_page_df.shape

## Common errors

### `No module named 'bs4'`

```bash
python3 -m pip install beautifulsoup4
```

### `AttributeError: 'NoneType' object has no attribute ...`

The selector found nothing. Inspect the element:

```python
element = book.select_one("your-selector")
print(element)
```

### Empty list

```python
books = soup.select("article.product_pod")
len(books)
```

If this returns `0`, the selector may be wrong, the page may have changed, or the content may be loaded with JavaScript.

### `KeyError: 'title'`

The selected element may not have a `title` attribute. Print it before extracting.

### Connection or timeout errors

Check the URL, your internet connection and the status code. Avoid rapid retry loops.


# Final recap

```text
URL
 ↓
requests.get()
 ↓
response.text
 ↓
BeautifulSoup()
 ↓
select repeated book cards
 ↓
extract fields
 ↓
list of dictionaries
 ↓
Pandas dataframe
 ↓
clean, validate and save
```